# LOAN23-02 GPU Fast CV Pipeline

Fast-runtime Kaggle notebook for loan approval classification with local-or-Kaggle data resolution, schema validation, GPU-first CatBoost training, stratified CV, OOF threshold tuning for F1, and submission-ready outputs.


In [ ]:
import importlib.util
import subprocess
import sys


def run_cmd(cmd):
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


def ensure_package(module_name, package_name=None):
    pkg = package_name or module_name
    if importlib.util.find_spec(module_name) is None:
        run_cmd([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("sklearn", "scikit-learn")
ensure_package("catboost")
ensure_package("kaggle")
ensure_package("dotenv", "python-dotenv")


In [ ]:
import os
import time
import warnings
from datetime import UTC, datetime
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from dotenv import load_dotenv
from IPython.display import display
from kaggle.api.kaggle_api_extended import KaggleApi
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)


In [ ]:
# Kaggle + notebook configuration
COMPETITION = "insurance-premium-prediction-cpe-232-data-models"
NOTEBOOK_SLUG = "loan23-02-gpu-fast-cv"

RUN_DOWNLOAD = True
RUN_SUBMISSION = True
FORCE_DOWNLOAD = True

CV_MODE = "stratified"
N_SPLITS = 5
SEED = 42
ENABLE_OPTUNA = False
ENABLE_ABLATIONS = True
ENABLE_ENSEMBLE = False

CATBOOST_ITERATIONS = 1500
EARLY_STOPPING_ROUNDS = 100
USE_GPU = True
GPU_DEVICE = "0"

ID_COL = "id"
TARGET_COL = "loan_status"
CATEGORICAL_COLS = ["education", "self_employed"]
NUMERIC_COLS = [
    "no_of_dependents",
    "income_annum",
    "loan_amount",
    "loan_term",
    "cibil_score",
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value",
]
MONEY_COLS = [
    "income_annum",
    "loan_amount",
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value",
]


def detect_gpu_available() -> bool:
    if not USE_GPU:
        return False
    try:
        proc = subprocess.run(
            ["nvidia-smi", "-L"],
            capture_output=True,
            text=True,
            timeout=5,
            check=False,
        )
        return proc.returncode == 0 and "GPU" in proc.stdout
    except Exception:
        return False


GPU_AVAILABLE = detect_gpu_available()
TRAIN_DEVICE_LABEL = "gpu" if GPU_AVAILABLE else "cpu"

OUTPUT_ROOT = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
)
ARTIFACT_DIR = OUTPUT_ROOT / "outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Output root:", OUTPUT_ROOT)
print("Artifacts dir:", ARTIFACT_DIR)
print("Training device:", TRAIN_DEVICE_LABEL)
print("CV mode:", CV_MODE)
print("CV folds:", N_SPLITS)


In [ ]:
def load_env_from_candidates():
    cwd = Path.cwd().resolve()
    candidates = [cwd / ".env", cwd.parent / ".env", Path("/kaggle/working/.env")]
    loaded = False
    for env_path in candidates:
        if env_path.exists():
            load_dotenv(env_path, override=False)
            print(f"Loaded env from {env_path}")
            loaded = True
            break
    if not loaded:
        print("No .env file found in default candidate locations.")


def resolve_local_data_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data",
        cwd / "notebooks/2026-03-20-loan-approval-prediction-cpe-232-data-models/data",
        cwd.parent / "data",
    ]
    for c in candidates:
        if c.exists() and (c / "train.csv").exists() and (c / "test.csv").exists():
            return c
    return cwd / "data"


def resolve_kaggle_input_dir(competition: str) -> Path | None:
    direct = Path("/kaggle/input") / competition
    if direct.exists() and (direct / "train.csv").exists():
        return direct

    if Path("/kaggle/input").exists():
        matches = sorted(Path("/kaggle/input").glob(f"*{competition}*"))
        for m in matches:
            if (m / "train.csv").exists() and (m / "test.csv").exists():
                return m
    return None


def ensure_competition_data(
    api: KaggleApi | None, competition: str, run_download: bool, force_download: bool
):
    kaggle_input = resolve_kaggle_input_dir(competition)
    if kaggle_input is not None and not run_download:
        print("Using mounted Kaggle input:", kaggle_input)
        return kaggle_input

    local_data = resolve_local_data_dir()
    local_data.mkdir(parents=True, exist_ok=True)

    train_path = local_data / "train.csv"
    test_path = local_data / "test.csv"
    sample_path = local_data / "sample_submission.csv"

    have_local = train_path.exists() and test_path.exists() and sample_path.exists()

    if run_download:
        if api is None:
            if not have_local:
                raise RuntimeError(
                    "RUN_DOWNLOAD=True but Kaggle auth unavailable and no local data found."
                )
            print("RUN_DOWNLOAD=True but API unavailable. Falling back to local files.")
            return local_data

        if force_download and local_data.exists():
            for z in local_data.glob("*.zip"):
                try:
                    z.unlink()
                except Exception:
                    pass

        if force_download or not have_local:
            print("Downloading competition files via Kaggle API...")
            api.competition_download_files(
                competition, path=str(local_data), force=force_download, quiet=False
            )

            zip_candidates = sorted(local_data.glob("*.zip"))
            if not zip_candidates:
                raise RuntimeError("Kaggle download did not produce a zip file.")
            latest_zip = max(zip_candidates, key=lambda p: p.stat().st_mtime)
            with ZipFile(latest_zip, "r") as zf:
                zf.extractall(local_data)
            print("Extracted:", latest_zip.name)

        have_local = train_path.exists() and test_path.exists() and sample_path.exists()
        if not have_local:
            raise RuntimeError(f"Data missing after download in {local_data}")

    if not have_local:
        raise RuntimeError(
            f"Data not found. Checked local dir {local_data}. "
            "Set RUN_DOWNLOAD=True with valid Kaggle credentials or provide local CSV files."
        )

    return local_data


load_env_from_candidates()
os.environ.pop("KAGGLE_API_TOKEN", None)

api = None
try:
    api = KaggleApi()
    api.authenticate()
    print("Authenticated user:", api.get_config_value("username"))
except Exception as auth_exc:
    print("Kaggle auth unavailable:", auth_exc)

data_dir = ensure_competition_data(
    api,
    COMPETITION,
    RUN_DOWNLOAD,
    FORCE_DOWNLOAD,
)
print("Resolved data dir:", data_dir)


In [ ]:
EXPECTED_TRAIN_COLUMNS = [
    "id",
    "no_of_dependents",
    "education",
    "self_employed",
    "income_annum",
    "loan_amount",
    "loan_term",
    "cibil_score",
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value",
    "loan_status",
]
EXPECTED_TEST_COLUMNS = [
    "id",
    "no_of_dependents",
    "education",
    "self_employed",
    "income_annum",
    "loan_amount",
    "loan_term",
    "cibil_score",
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value",
]


def fail_schema(message: str):
    raise ValueError(f"[SCHEMA ERROR] {message}")


def load_and_validate_data(data_dir: Path):
    train_path = data_dir / "train.csv"
    test_path = data_dir / "test.csv"
    sample_path = data_dir / "sample_submission.csv"

    if not train_path.exists() or not test_path.exists() or not sample_path.exists():
        fail_schema("Expected train.csv, test.csv, and sample_submission.csv to exist.")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    sample_df = pd.read_csv(sample_path)

    train_df.columns = [str(c).strip() for c in train_df.columns]
    test_df.columns = [str(c).strip() for c in test_df.columns]
    sample_df.columns = [str(c).strip() for c in sample_df.columns]

    if train_df.columns.tolist() != EXPECTED_TRAIN_COLUMNS:
        fail_schema(
            f"Unexpected train columns. expected={EXPECTED_TRAIN_COLUMNS}, got={train_df.columns.tolist()}"
        )
    if test_df.columns.tolist() != EXPECTED_TEST_COLUMNS:
        fail_schema(
            f"Unexpected test columns. expected={EXPECTED_TEST_COLUMNS}, got={test_df.columns.tolist()}"
        )
    if sample_df.columns.tolist() != [ID_COL, TARGET_COL]:
        fail_schema(
            f"Unexpected sample_submission columns. expected={[ID_COL, TARGET_COL]}, got={sample_df.columns.tolist()}"
        )

    train_features = [c for c in train_df.columns if c != TARGET_COL]
    if train_features != test_df.columns.tolist():
        fail_schema("Train/test feature mismatch after removing target column.")

    for col in NUMERIC_COLS + [TARGET_COL]:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
    for col in NUMERIC_COLS:
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

    if not set(train_df[TARGET_COL].dropna().unique()).issubset({0, 1}):
        fail_schema("Target column must contain only 0/1 values.")

    if not sample_df[ID_COL].isin(test_df[ID_COL]).all():
        print(
            "Warning: sample_submission ids differ from test ids. Submission will use test ids."
        )

    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)
    print("Sample shape:", sample_df.shape)
    print("Target balance:")
    display(
        train_df[TARGET_COL].value_counts(normalize=True).sort_index().rename("ratio")
    )
    print("Missing values (train):")
    display(train_df.isna().sum().sort_values(ascending=False).rename("missing_train"))

    return train_df, test_df, sample_df


train_df, test_df, sample_submission = load_and_validate_data(data_dir)
print("Schema ready.")


In [ ]:
MISSING_TOKENS = {"", "na", "n/a", "none", "null", "nan", "unknown", "missing", "?"}
FEATURE_CACHE = {}


def normalize_text_like(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    low = s.str.lower()
    return s.mask(low.isin(MISSING_TOKENS), pd.NA)


def robust_numeric(series: pd.Series) -> pd.Series:
    out = pd.to_numeric(series, errors="coerce")
    return out.replace([np.inf, -np.inf], np.nan)


def clean_base_frames(train_raw: pd.DataFrame, test_raw: pd.DataFrame):
    tr = train_raw.copy(deep=True)
    te = test_raw.copy(deep=True)

    for col in NUMERIC_COLS:
        tr[col] = robust_numeric(tr[col])
        te[col] = robust_numeric(te[col])

    tr[TARGET_COL] = robust_numeric(tr[TARGET_COL]).astype("int8")

    for col in CATEGORICAL_COLS:
        tr[col] = normalize_text_like(tr[col])
        te[col] = normalize_text_like(te[col])

    return tr, te


def fill_numeric_median(tr: pd.DataFrame, te: pd.DataFrame, columns: list[str]):
    for col in columns:
        med = float(tr[col].median()) if tr[col].notna().any() else 0.0
        tr[col] = tr[col].fillna(med)
        te[col] = te[col].fillna(med)


def fill_categorical_missing(tr: pd.DataFrame, te: pd.DataFrame, columns: list[str]):
    for col in columns:
        tr[col] = tr[col].fillna("Missing").astype("string")
        te[col] = te[col].fillna("Missing").astype("string")

        vocab = sorted(set(tr[col].dropna().astype(str).unique()))
        if "Unknown" not in vocab:
            vocab.append("Unknown")
        tr[col] = tr[col].where(tr[col].isin(vocab), "Unknown")
        te[col] = te[col].where(te[col].isin(vocab), "Unknown")


def add_missing_indicators(tr: pd.DataFrame, te: pd.DataFrame, columns: list[str]):
    created = []
    for col in columns:
        if tr[col].isna().mean() <= 0:
            continue
        name = f"mis_{col}"
        tr[name] = tr[col].isna().astype("int8")
        te[name] = te[col].isna().astype("int8")
        created.append(name)
    return created


def add_extended_features(tr: pd.DataFrame, te: pd.DataFrame):
    asset_cols = [
        "residential_assets_value",
        "commercial_assets_value",
        "luxury_assets_value",
        "bank_asset_value",
    ]

    tr["total_assets"] = tr[asset_cols].sum(axis=1)
    te["total_assets"] = te[asset_cols].sum(axis=1)

    loan_safe_tr = tr["loan_amount"].replace(0, np.nan)
    loan_safe_te = te["loan_amount"].replace(0, np.nan)
    income_safe_tr = tr["income_annum"].replace(0, np.nan)
    income_safe_te = te["income_annum"].replace(0, np.nan)
    term_safe_tr = tr["loan_term"].replace(0, np.nan)
    term_safe_te = te["loan_term"].replace(0, np.nan)

    tr["total_assets_to_loan"] = (tr["total_assets"] / loan_safe_tr).replace(
        [np.inf, -np.inf], np.nan
    )
    te["total_assets_to_loan"] = (te["total_assets"] / loan_safe_te).replace(
        [np.inf, -np.inf], np.nan
    )
    tr["income_to_loan"] = (tr["income_annum"] / loan_safe_tr).replace(
        [np.inf, -np.inf], np.nan
    )
    te["income_to_loan"] = (te["income_annum"] / loan_safe_te).replace(
        [np.inf, -np.inf], np.nan
    )
    tr["loan_to_income"] = (tr["loan_amount"] / income_safe_tr).replace(
        [np.inf, -np.inf], np.nan
    )
    te["loan_to_income"] = (te["loan_amount"] / income_safe_te).replace(
        [np.inf, -np.inf], np.nan
    )
    tr["loan_per_term"] = (tr["loan_amount"] / term_safe_tr).replace(
        [np.inf, -np.inf], np.nan
    )
    te["loan_per_term"] = (te["loan_amount"] / term_safe_te).replace(
        [np.inf, -np.inf], np.nan
    )
    tr["income_per_dependent"] = tr["income_annum"] / (
        tr["no_of_dependents"].fillna(0) + 1.0
    )
    te["income_per_dependent"] = te["income_annum"] / (
        te["no_of_dependents"].fillna(0) + 1.0
    )
    tr["assets_minus_loan"] = tr["total_assets"] - tr["loan_amount"].fillna(0)
    te["assets_minus_loan"] = te["total_assets"] - te["loan_amount"].fillna(0)

    total_assets_safe_tr = tr["total_assets"].replace(0, np.nan)
    total_assets_safe_te = te["total_assets"].replace(0, np.nan)
    for col in asset_cols:
        ratio_name = f"{col}_share"
        tr[ratio_name] = (tr[col] / total_assets_safe_tr).replace(
            [np.inf, -np.inf], np.nan
        )
        te[ratio_name] = (te[col] / total_assets_safe_te).replace(
            [np.inf, -np.inf], np.nan
        )

    for col in MONEY_COLS + ["total_assets"]:
        tr[f"log1p_{col}"] = np.log1p(np.clip(tr[col].fillna(0), a_min=0, a_max=None))
        te[f"log1p_{col}"] = np.log1p(np.clip(te[col].fillna(0), a_min=0, a_max=None))

    tr["cibil_band"] = pd.cut(
        tr["cibil_score"],
        bins=[-np.inf, 500, 650, 750, 850, np.inf],
        labels=["very_low", "low", "mid", "high", "very_high"],
    ).astype("string")
    te["cibil_band"] = pd.cut(
        te["cibil_score"],
        bins=[-np.inf, 500, 650, 750, 850, np.inf],
        labels=["very_low", "low", "mid", "high", "very_high"],
    ).astype("string")

    tr["income_band"] = pd.cut(
        tr["income_annum"],
        bins=[-np.inf, 1000000, 3000000, 6000000, 9000000, np.inf],
        labels=["very_low", "low", "mid", "high", "very_high"],
    ).astype("string")
    te["income_band"] = pd.cut(
        te["income_annum"],
        bins=[-np.inf, 1000000, 3000000, 6000000, 9000000, np.inf],
        labels=["very_low", "low", "mid", "high", "very_high"],
    ).astype("string")

    tr["loan_band"] = pd.cut(
        tr["loan_amount"],
        bins=[-np.inf, 5000000, 10000000, 20000000, 30000000, np.inf],
        labels=["very_low", "low", "mid", "high", "very_high"],
    ).astype("string")
    te["loan_band"] = pd.cut(
        te["loan_amount"],
        bins=[-np.inf, 5000000, 10000000, 20000000, 30000000, np.inf],
        labels=["very_low", "low", "mid", "high", "very_high"],
    ).astype("string")


def finalize_frame_types(tr: pd.DataFrame, te: pd.DataFrame, feature_cols: list[str]):
    cat_cols = []
    for col in feature_cols:
        if pd.api.types.is_numeric_dtype(tr[col]):
            tr[col] = robust_numeric(tr[col])
            te[col] = robust_numeric(te[col])
        else:
            tr[col] = normalize_text_like(tr[col]).fillna("Missing").astype("string")
            te[col] = normalize_text_like(te[col]).fillna("Missing").astype("string")
            vocab = sorted(set(tr[col].dropna().astype(str).unique()))
            if "Unknown" not in vocab:
                vocab.append("Unknown")
            tr[col] = tr[col].where(tr[col].isin(vocab), "Unknown").astype("string")
            te[col] = te[col].where(te[col].isin(vocab), "Unknown").astype("string")
            cat_cols.append(col)
    return cat_cols


def build_features(variant: str, train_raw: pd.DataFrame, test_raw: pd.DataFrame):
    if variant not in {"raw_minimal", "processed_basic", "processed_extended"}:
        raise ValueError(f"Unknown feature variant: {variant}")

    tr, te = clean_base_frames(train_raw, test_raw)

    if variant == "raw_minimal":
        fill_numeric_median(tr, te, NUMERIC_COLS)
        fill_categorical_missing(tr, te, CATEGORICAL_COLS)
    elif variant == "processed_basic":
        add_missing_indicators(tr, te, NUMERIC_COLS + CATEGORICAL_COLS)
        fill_numeric_median(tr, te, NUMERIC_COLS)
        fill_categorical_missing(tr, te, CATEGORICAL_COLS)
    elif variant == "processed_extended":
        add_missing_indicators(tr, te, NUMERIC_COLS + CATEGORICAL_COLS)
        fill_numeric_median(tr, te, NUMERIC_COLS)
        fill_categorical_missing(tr, te, CATEGORICAL_COLS)
        add_extended_features(tr, te)

    feature_cols = [c for c in tr.columns if c not in {ID_COL, TARGET_COL}]
    cat_cols = finalize_frame_types(tr, te, feature_cols)

    meta = {
        "feature_cols": feature_cols,
        "cat_cols": cat_cols,
    }
    return tr, te, meta


def get_feature_bundle(feature_variant: str):
    if feature_variant not in FEATURE_CACHE:
        FEATURE_CACHE[feature_variant] = build_features(
            feature_variant, train_df, test_df
        )
        print(f"[cache] built features for {feature_variant}")
    return FEATURE_CACHE[feature_variant]


In [ ]:
def make_folds(y: pd.Series, n_splits: int, seed: int):
    idx = np.arange(len(y))
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []
    for tr_idx, va_idx in splitter.split(idx, y):
        folds.append((tr_idx, va_idx))

    fold_id = np.full(len(y), -1, dtype=np.int16)
    for fold, (_, va_idx) in enumerate(folds):
        fold_id[va_idx] = fold
    if (fold_id < 0).any():
        raise RuntimeError("Some rows were not assigned to folds.")
    return folds, fold_id


def build_model(
    seed: int, params_override: dict | None = None, force_cpu: bool = False
):
    params_override = params_override or {}
    use_gpu = USE_GPU and GPU_AVAILABLE and not force_cpu
    base = {
        "loss_function": "Logloss",
        "eval_metric": "F1",
        "learning_rate": 0.03,
        "depth": 8,
        "l2_leaf_reg": 5.0,
        "iterations": CATBOOST_ITERATIONS,
        "random_seed": seed,
        "allow_writing_files": False,
        "verbose": False,
    }
    if use_gpu:
        base.update({"task_type": "GPU", "devices": GPU_DEVICE})
    base.update(params_override)
    return CatBoostClassifier(**base)


def fit_predict_one_fold(model, X_tr, y_tr, X_va, y_va, X_te, cat_cols: list[str]):
    cat_idx = [X_tr.columns.get_loc(c) for c in cat_cols if c in X_tr.columns]
    model.fit(
        X_tr,
        y_tr,
        eval_set=(X_va, y_va),
        cat_features=cat_idx if cat_idx else None,
        use_best_model=True,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose=False,
    )
    pred_va = model.predict_proba(X_va)[:, 1]
    pred_te = model.predict_proba(X_te)[:, 1]
    return pred_va, pred_te


def extract_feature_importance(model, feature_names: list[str]):
    try:
        vals = model.get_feature_importance()
        return pd.DataFrame({"feature": feature_names, "importance": vals})
    except Exception:
        return pd.DataFrame(columns=["feature", "importance"])


FOLDS, FOLD_ID = make_folds(train_df[TARGET_COL], N_SPLITS, SEED)
fold_index_path = ARTIFACT_DIR / f"fold_indices_{NOTEBOOK_SLUG}.csv"
pd.DataFrame({ID_COL: train_df[ID_COL], "fold_id": FOLD_ID}).to_csv(
    fold_index_path, index=False
)
print("Saved fold assignments to", fold_index_path)


In [ ]:
EXP_ROWS = []
OOF_STORE = {}
TEST_STORE = {}
FI_ROWS = []
THRESHOLD_ROWS = []


def threshold_search_rows(y_true: np.ndarray, pred_proba: np.ndarray, experiment: str):
    rows = []
    thresholds = np.round(np.linspace(0.05, 0.95, 181), 4)
    for thr in thresholds:
        pred_label = (pred_proba >= float(thr)).astype(int)
        rows.append(
            {
                "experiment": experiment,
                "threshold": float(thr),
                "f1": float(f1_score(y_true, pred_label)),
                "precision": float(
                    precision_score(y_true, pred_label, zero_division=0)
                ),
                "recall": float(recall_score(y_true, pred_label, zero_division=0)),
            }
        )
    return pd.DataFrame(rows)


def pick_best_threshold(search_df: pd.DataFrame):
    ranked = search_df.sort_values(
        ["f1", "precision", "recall", "threshold"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    return ranked.iloc[0]


def run_experiment(exp_config: dict):
    name = exp_config["name"]
    feature_variant = exp_config["feature_variant"]
    params_override = exp_config.get("params_override") or {}
    seed = int(exp_config.get("seed", SEED))

    t0 = time.time()
    tr, te, meta = get_feature_bundle(feature_variant)
    feature_cols = meta["feature_cols"]
    cat_cols = meta["cat_cols"]

    X = tr[feature_cols].copy()
    X_test = te[feature_cols].copy()
    y = tr[TARGET_COL].to_numpy(dtype=int)

    oof_pred = np.zeros(len(tr), dtype=float)
    test_fold_preds = []
    fold_f1s = []
    fold_precisions = []
    fold_recalls = []
    force_cpu = False

    print(
        f"\n[{name}] start | variant={feature_variant} folds={len(FOLDS)} device={TRAIN_DEVICE_LABEL}"
    )
    for fold, (idx_tr, idx_va) in enumerate(FOLDS):
        fold_t0 = time.time()
        X_tr = X.iloc[idx_tr].copy()
        y_tr = y[idx_tr]
        X_va = X.iloc[idx_va].copy()
        y_va = y[idx_va]

        model = build_model(
            seed + fold, params_override=params_override, force_cpu=force_cpu
        )
        try:
            pred_va, pred_te = fit_predict_one_fold(
                model, X_tr, y_tr, X_va, y_va, X_test, cat_cols
            )
        except Exception as exc:
            if not force_cpu and GPU_AVAILABLE:
                print(
                    f"[{name}] fold {fold + 1}/{len(FOLDS)} GPU failed ({type(exc).__name__}), retrying on CPU."
                )
                force_cpu = True
                model = build_model(
                    seed + fold, params_override=params_override, force_cpu=True
                )
                pred_va, pred_te = fit_predict_one_fold(
                    model, X_tr, y_tr, X_va, y_va, X_test, cat_cols
                )
            else:
                raise

        oof_pred[idx_va] = pred_va
        test_fold_preds.append(pred_te)

        pred_label_05 = (pred_va >= 0.5).astype(int)
        fold_f1 = float(f1_score(y_va, pred_label_05))
        fold_precision = float(precision_score(y_va, pred_label_05, zero_division=0))
        fold_recall = float(recall_score(y_va, pred_label_05, zero_division=0))
        fold_f1s.append(fold_f1)
        fold_precisions.append(fold_precision)
        fold_recalls.append(fold_recall)

        fi = extract_feature_importance(model, feature_cols)
        if len(fi):
            fi["experiment"] = name
            fi["fold"] = fold
            FI_ROWS.append(fi)

        fold_elapsed = float(time.time() - fold_t0)
        print(
            f"[{name}] fold {fold + 1}/{len(FOLDS)} f1@0.5={fold_f1:.6f} "
            f"precision={fold_precision:.6f} recall={fold_recall:.6f} time={fold_elapsed:.1f}s"
        )

    test_pred = np.mean(np.vstack(test_fold_preds), axis=0)
    search_df = threshold_search_rows(y, oof_pred, name)
    THRESHOLD_ROWS.extend(search_df.to_dict(orient="records"))
    best_thr_row = pick_best_threshold(search_df)

    oof_label_05 = (oof_pred >= 0.5).astype(int)
    oof_label_best = (oof_pred >= float(best_thr_row["threshold"])).astype(int)

    elapsed = float(time.time() - t0)
    device_used = "cpu-fallback" if force_cpu else TRAIN_DEVICE_LABEL
    EXP_ROWS.append(
        {
            "experiment": name,
            "model_family": "catboost",
            "feature_variant": feature_variant,
            "device": device_used,
            "best_threshold": float(best_thr_row["threshold"]),
            "cv_f1_mean_05": float(np.mean(fold_f1s)),
            "cv_f1_std_05": float(np.std(fold_f1s)),
            "cv_precision_mean_05": float(np.mean(fold_precisions)),
            "cv_recall_mean_05": float(np.mean(fold_recalls)),
            "oof_f1_05": float(f1_score(y, oof_label_05)),
            "oof_f1_best": float(f1_score(y, oof_label_best)),
            "oof_precision_best": float(
                precision_score(y, oof_label_best, zero_division=0)
            ),
            "oof_recall_best": float(recall_score(y, oof_label_best, zero_division=0)),
            "train_time_sec": elapsed,
            "n_features": len(feature_cols),
        }
    )

    OOF_STORE[name] = oof_pred
    TEST_STORE[name] = test_pred
    print(
        f"[{name}] oof_f1@0.5={f1_score(y, oof_label_05):.6f} "
        f"oof_f1@best={f1_score(y, oof_label_best):.6f} "
        f"best_threshold={float(best_thr_row['threshold']):.4f} time={elapsed:.1f}s device={device_used}"
    )
    return name


BASE_EXPERIMENTS = [
    {
        "name": "catboost_processed_basic",
        "feature_variant": "processed_basic",
    },
    {
        "name": "catboost_processed_extended",
        "feature_variant": "processed_extended",
    },
]

if ENABLE_ABLATIONS:
    BASE_EXPERIMENTS.insert(
        0,
        {
            "name": "catboost_raw_minimal",
            "feature_variant": "raw_minimal",
        },
    )

for exp in BASE_EXPERIMENTS:
    run_experiment(exp)

leaderboard_df = (
    pd.DataFrame(EXP_ROWS)
    .sort_values(["oof_f1_best", "cv_f1_mean_05"], ascending=[False, False])
    .reset_index(drop=True)
)
display(leaderboard_df)


In [ ]:
results_df = (
    pd.DataFrame(EXP_ROWS)
    .sort_values(["oof_f1_best", "cv_f1_mean_05"], ascending=[False, False])
    .reset_index(drop=True)
)
display(results_df)

exp_path = ARTIFACT_DIR / f"experiments_{NOTEBOOK_SLUG}.csv"
results_df.to_csv(exp_path, index=False)

threshold_df = (
    pd.DataFrame(THRESHOLD_ROWS)
    .sort_values(["experiment", "f1", "threshold"], ascending=[True, False, True])
    .reset_index(drop=True)
)
threshold_path = ARTIFACT_DIR / f"thresholds_{NOTEBOOK_SLUG}.csv"
threshold_df.to_csv(threshold_path, index=False)

best_exp = results_df.iloc[0]["experiment"]
best_row = results_df.iloc[0]
best_oof = OOF_STORE[best_exp]
best_test = TEST_STORE[best_exp]
best_threshold = float(best_row["best_threshold"])

oof_df = pd.DataFrame(
    {
        ID_COL: train_df[ID_COL],
        "y_true": train_df[TARGET_COL].astype(int),
        "pred_proba": best_oof,
        "pred_label_05": (best_oof >= 0.5).astype(int),
        "pred_label_best": (best_oof >= best_threshold).astype(int),
        "experiment": best_exp,
    }
)
for c in CATEGORICAL_COLS + NUMERIC_COLS:
    if c in train_df.columns and c not in oof_df.columns:
        oof_df[c] = train_df[c]

oof_path = ARTIFACT_DIR / f"oof_predictions_{NOTEBOOK_SLUG}.csv"
oof_df.to_csv(oof_path, index=False)

if FI_ROWS:
    fi_df = pd.concat(FI_ROWS, ignore_index=True)
    fi_summary = (
        fi_df.groupby(["experiment", "feature"], as_index=False)["importance"]
        .mean()
        .sort_values(["experiment", "importance"], ascending=[True, False])
    )
else:
    fi_summary = pd.DataFrame(columns=["experiment", "feature", "importance"])

fi_path = ARTIFACT_DIR / f"feature_importance_{NOTEBOOK_SLUG}.csv"
fi_summary.to_csv(fi_path, index=False)

submission_df = pd.DataFrame(
    {
        ID_COL: test_df[ID_COL].astype(int),
        TARGET_COL: (best_test >= best_threshold).astype(int),
    }
)
submission_df = submission_df[[ID_COL, TARGET_COL]]

sub_path = OUTPUT_ROOT / f"submission_{NOTEBOOK_SLUG}.csv"
submission_df.to_csv(sub_path, index=False)

print("Saved experiments:", exp_path)
print("Saved thresholds:", threshold_path)
print("Saved OOF predictions:", oof_path)
print("Saved feature importance:", fi_path)
print("Saved submission:", sub_path)
print()
print("Recommended model:", best_exp)
print("OOF F1 @ 0.5:", round(float(best_row["oof_f1_05"]), 6))
print("OOF F1 @ best:", round(float(best_row["oof_f1_best"]), 6))
print("Best threshold:", round(best_threshold, 4))
print("CV F1 mean @ 0.5:", round(float(best_row["cv_f1_mean_05"]), 6))
print("CV F1 std @ 0.5:", round(float(best_row["cv_f1_std_05"]), 6))


In [ ]:
if RUN_SUBMISSION:
    if api is None:
        raise RuntimeError("RUN_SUBMISSION=True but Kaggle API auth is not available")

    submit_message = (
        f"{NOTEBOOK_SLUG} | "
        f"cv_mode={CV_MODE} n_splits={N_SPLITS} seed={SEED} | "
        f"time={datetime.now(UTC).isoformat()}"
    )
    api.competition_submit(
        file_name=str(OUTPUT_ROOT / f"submission_{NOTEBOOK_SLUG}.csv"),
        message=submit_message,
        competition=COMPETITION,
    )
    print("Submission sent:", submit_message)
